# Tokenizer
We started to experiment with `Jin Yong`

In [1]:
import requests
import pandas as pd
from pathlib import Path
import json
import regex
import time

In [60]:
REGEX_PATTERN = r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"

In [61]:
pat = regex.compile(
    REGEX_PATTERN
    )

In [3]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

In [4]:
CORPUS = "jy"
CORPUS_DATA = get_locations(CORPUS)

In [9]:
text = "\n".join(list(f.read_text() for f  in (CORPUS_DATA.parent.parent / CORPUS).rglob("*.txt")))

In [10]:
text[:100]

'\n# chapter:六\n## book:越女剑(旧版)\n\n\u3000\u3000阿青竹棒一动，对手若不是手腕被戳，长剑脱手，那便是要害中招，委顿在地。\n\n\u3000\u3000第二天是三十名剑士败在她的手下，第三天又是三十名剑士在她一'

In [11]:
len(text)

17354146

In [12]:
utf8 = list(text.encode())

In [13]:
len(utf8)

51605217

In [14]:
freq = dict()

for c1, c2 in zip(utf8,utf8[1:]):
    freq[(c1, c2)] = freq.get((c1, c2),0) + 1

In [15]:
most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])
most_freq[:5]

[((239, 188), 1569623),
 ((188, 140), 1227397),
 ((228, 184), 1162859),
 ((227, 128), 621391),
 ((228, 186), 533384)]

In [47]:
# code_list = list(utf8)

code_collection = list(list(sub_text.encode("utf8")) for sub_text in pat.findall(text))

codes_start = 0
utf_len = len(list(text.encode("utf8")))
for l in code_collection:

    codes_start = max(max(l), codes_start)

vocab = dict()


def merge(ids, temp_vocab):
    if len(ids) < 2:
        return ids
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        token = temp_vocab.get((ids[i], ids[i + 1]))
        if token is not None:
            new_ids.append(token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

logs = []

last_ts = time.time()
while True:
    freq = dict()

    for code_list in code_collection:

        if len(code_list) < 2:
            continue

        for c1, c2 in zip(code_list, code_list[1:]):
            freq[(c1, c2)] = freq.get((c1, c2), 0) + 1

    most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])

    temp_vocab = dict()
    top_freq_ids1 = set()
    top_freq_ids2 = set()

    for (c1, c2), ct in most_freq:
        if c1 in top_freq_ids2 or c2 in top_freq_ids1 or ct < 5:
            break
        top_freq_ids1.add(c1)
        top_freq_ids2.add(c2)

        codes_start += 1

        temp_vocab[(c1, c2)] = codes_start

        vocab[codes_start] = (c1, c2)

        if len(vocab) >= 30_000:
            break

    # print(temp_vocab)

    if len(temp_vocab) == 0:
        print(f"\nFinished Vocab:{len(vocab)} /{ct} / {ct/utf_len:.5f}")
        break

    # print(f"Try to merge {len(temp_vocab)}")
    new_code = codes_start

    if len(vocab) % 100 == 0:
        print(f"V:{len(vocab)}x", end="\t")

    ct = 0

    for i in range(len(code_collection)):
        code_collection[i] = merge(code_collection[i], temp_vocab)
        ct += len(code_collection[i])

    new_ts = time.time()

    logs.append({
        "vocab_len": len(vocab),
        "code_list_len": ct,
        "compression": ct / utf_len,
        "time": new_ts - last_ts,
    })

    last_ts = new_ts

    if len(vocab) >= 30_000:
        break
    

V:3100x	V:4000x	V:4300x	V:4400x	V:10300x	V:13100x	V:14100x	V:16600x	V:19400x	V:20200x	V:21500x	V:29400x	V:30000x	

In [57]:
import sys

In [58]:
sys.path.append("../src")

In [59]:
from tokenizer import train_tokenizer

In [79]:
vocab, logs = train_tokenizer(text)

In [81]:
df = pd.DataFrame(logs)
df

,code_list_len,compression,time,vocab_len
0,50035594.0,0.969584,0.261596,1.0
1,45957633.0,0.890562,0.295203,6.0
2,42571598.0,0.824948,0.304255,17.0
3,41389525.0,0.802041,0.252266,22.0
4,37187027.0,0.720606,0.452999,46.0
...,...,...,...,...
1139,10581137.0,0.205040,0.097951,29919.0
1140,10580257.0,0.205023,0.049906,29939.0
1141,10578453.0,0.204988,0.075870,29980.0
1142,10577925.0,0.204978,0.058224,29992.0


In [82]:
with open(CORPUS_DATA / "vocab.json", "w") as f:
    f.write(json.dumps(vocab, indent=2), )

with open(CORPUS_DATA / "freq.json", "w") as f:
    f.write(json.dumps(most_freq, indent=2))

In [83]:
df = pd.DataFrame(logs)
df.to_csv(str(CORPUS_DATA / "logs.csv"), index=False)

## Retro Analysis

In [84]:
import json
import pandas as pd

In [85]:
df = pd.read_csv(str(CORPUS_DATA / "logs.csv"))

In [86]:
with open(CORPUS_DATA/"vocab.json") as f:
    vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())

with open(CORPUS_DATA/"freq.json") as f:
    most_freq = json.loads(f.read())

In [87]:
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_df(df):
    cols = ["vocab_len", "code_list_len", "compression", "time"]
    titles = ["Vocab Size", "Code List Length", "Compression Ratio", "Time per Step (s)"]

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

    for i, (col, title) in enumerate(zip(cols, titles)):
        row, col_idx = divmod(i, 2)
        fig.add_trace(
            go.Scatter(x=df.index, y=df[col], mode="lines", name=title),
            row=row + 1, col=col_idx + 1,
        )

    fig.update_layout(height=600, title_text="BPE Tokenizer Training", showlegend=False)
    fig.show()


In [88]:
plot_df(df)

In [70]:
pair_to_code = dict((tuple(v), int(k)) for k, v in vocab.items())

In [71]:
from loguru import logger

In [72]:
def encode(text: str, pair_to_code):
    code_list = list(text.encode("utf8"))
    if len(code_list) < 2:
        return code_list
    while len(code_list) > 1:
        temp_vocab = dict()
        for c1, c2 in zip(code_list, code_list[1:]):
            token = pair_to_code.get((c1, c2))
            if token is None:
                continue
            else:
                temp_vocab[(c1, c2)] = token
        if len(temp_vocab) == 0:
            return code_list


        pair = min(temp_vocab, key=lambda x: temp_vocab[x])
        token = temp_vocab[pair]

        new_code_list = []

        i = 0
        while i < len(code_list):
            if i < len(code_list) - 1 and code_list[i] == pair[0] and code_list[i + 1] == pair[1]:
                logger.info(f"[REPLACE]{code_list[i]}, {code_list[i + 1]} -> {token}")
                new_code_list.append(token)
                i += 2
            else:
                new_code_list.append(code_list[i])
                i += 1
        
        code_list = new_code_list
            
    return code_list

In [73]:
list("华山派".encode("utf8"))

[229, 141, 142, 229, 177, 177, 230, 180, 190]

In [74]:
token_ids = encode("华山派", pair_to_code=pair_to_code)
print(token_ids)

[2598]


In [75]:
token_ids = encode("五岳剑派同气连枝", pair_to_code=pair_to_code)
print(token_ids)

[5194, 27295]


In [76]:
def decode(tokens, vocab):
    result = list(tokens)
    
    while True:
        i = 0
        new_result = []
        temp_vocab = dict((i, vocab[i]) for i in result if i in vocab)
        if len(temp_vocab) == 0:
            break
        token = max(temp_vocab, key=lambda x: x)
        c1, c2 = temp_vocab[token]
        for t in result:
            if t == token:
                new_result.append(c1)
                new_result.append(c2)
            else:
                new_result.append(t)
        result = new_result
        
    return bytes(result).decode(errors="replace")


def by_token_decode(tokens, vocab):
    results = []
    for token in tokens:
        by_token_str = decode([token], vocab)
        results.append(by_token_str)
    return results
    

In [77]:
for i in range(1000, 2000):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else ", ")

吧, 见他, 脸上, 身上, 华, 之间, �, 计, 纵, 送, 哪, 机, 孩, 树, 莫, 异, 辈, 斐, 胸, 这里, �
义, 承志, 痛, 房, 胡斐, �, 料, 不可, 摇, �, 康, 湖, 件, 登, �, 攻, 却是, 找, 一声, 至, 今日, 虽然, 晚, 功夫, 厉, 没有, �, 果, 错, 少林
可是, 北, 陆, 片, �, 出去, 侠, 帝, 礼, 感, 种, 闪, 寻, 仇, 朱, 船, 吴, 因, 精, 大哥, 追, 夫人, 一人, 丁, 仍, 举, 香, 这个, 拍, 骂
天下, 罢, �, 麽, 寺, 劲, 两个, 逃, 宫, 刚, 见到, 伸手, �, �, 底, 拉, 之下, 说话, 抢, 左手, 报, 久, 沉, 过去, �, 这些, 鬼, 息, 　那, 流
推, 之后, 缓, 紫, 这么, 隐, 居, 顶, 求, 萧, �, 若是, 三人, 你的, 姊, �, 喝道, 根, 领, 正是, 禁, 还是, �, 妙, 下来, 有人, 失, 声音, 盈, 在地
教主, 洛, 现, 韦小宝道, 登时, 岂, 抱, 均, 银, 爱, 住了, 震, 月, 易, 卫, 另, 拜, 手中, 避, 有一, 活, 施,  
, 性命, 每, 妇, 合, 大师, 我的, 玄
化, 任, 复, 险, 离, 就是, 包, 曾, 耳, 一笑, 肯, �, 爹爹, 那人, 怀, 猛, 赶, 长剑, 星, 了几, 慕, 当真, 刃, 公子, 记, 又是, 仙, 完, 留, 屋
丐, 挥, �, 那是, 句话, �, 徒, 洞, 战, 家洛, 英雄, 友, 十分, 双手, 陈家洛, 细, 贼, 上一, 形, 骨, 问道, 过来, 这位, 敬, 腿, 渐, 威, 眼见, 欧, 龙女
肩, 显, 谷, �, 锋, 小龙女, 欢, 人家, 不到, 倘, �, 欧阳, 袁承志, 跳, 他一, �, 孙, 部, 蒙, 横, 归, 後, 右手, 雪, 识, 毫, 这样, 心下, 跟着, 母
露, 守, 不由, 呢, 称, 中的, 美, 剑法, 游, 块, �, 免, 乎, 败, 难道, 结, 收, 罪, 哈哈, 山派, 人的, 保, �, 江湖, 智, �, 不会, 乘, 自然, 只怕
点头, 则, 脱, 叹, 殷, 翻, 叔, 田, 管, 皇帝, 丝, 忽

In [78]:
for i in range(29000, 30000):
    try:
        print(
            decode([i], vocab=vocab),
            end="\n" if i % 30 == 0 else "| ")
    except ValueError:
        print(i)

张去| 一只小| 未始| 分际| 真功夫| 亏得| 从来不敢| 吹气| 做这| 真想| 学武功
富家| 他们三人| 缭乱| 称雄| 的一间| 横行天下| 疼爱| 愠道| 绑了| 说个明白| 佩服得五体投地| 第二件事| 让我们| 又走| 来到一处| 胚| 　这一招| 猛见| 睡到中夜| 霍霍| 光杰| 个屁| 当真有| 识破了| 移到| 不慎| 的功劳| 一挣| 他如此| 锏
都笑了起来| 但咱们| 略定| 师伯师叔| 疾攻| 个鬼脸| 一提马缰| 踞| 下不了手| 他眼见| 十二年| 美貌女子| 二庄主| 情若| 但当| 知道只要| 待死| 高高矮矮| 打成平手| 田伯父| 你有甚么| 接掌| 宗派| 下酒| 许久| 大擒拿手| 下台| 嗤的一响| 方有德| 　周仲英
陈家洛把| 张扬| 玄功| 军门| 就由| 郡王| 和宫| 顶红| 心中一宽| 众镖师| 满门抄| 满门抄斩| 这一番话| 嵩阳| 厩| 好极啦| 云贵| 众英雄| 难以索解| 皇上要| 这六人| 霭| 人吗| 配制| 葡�| 葡萄| 桑拉| 桑拉巴| 兀鹰| 圆性道
便把| 和人动手| 日后再| 忽哨| 火叉| 你在那里| 和阿珂| 黑龙使| 陆高轩道| 练内功| 韦公子| 水银| 上崖来| 郑王爷| 　韦小宝心道| 太妃| 南怀仁| 吴应熊这小子| 徐三哥| 半部| 少奶奶| 不老| 石碑| 穆尔| 衮| 岳武穆| 华伯斯基| 拔都| 刘贤弟| 掌门师兄
令狐冲微笑道| 仪琳师妹| 总坛| 教规| 　林平之道| 我华山派| 　劳德诺| 双戟| 毒蛛| 壸| 　谢逊道| 明教中人| 殷无福| 金刚伏魔| 怪蛇| 呼延万| 呼延万善| 　石中玉| 萧半和| 　陆冠英| 陆庄主道| 郭杨二人| 全金发道| 但小龙女| 给杨过| 　尼摩星| 　忽必烈| 武哥哥| 尹赵二人| 綫
这个小姑娘| 喜欢我| 发出的| 蹲在地下| 有大| 卉| 他总是| 讲述| 映在| 白色的| 一声大喝| 微微躬身| 闪闪生光| 中宫| 脸上却是| 又是一剑| 亡国| 美色| 承蒙| 后面跟着| 不免有些| 一大块| 毫不理睬| 高傲| 劝解| 那白衣| 那我就| 跟我说了| 打劫| 领略
不知其| 辨明| 黄土| 有话好说| 赤身| 也用| 生气了| 从此不再| 还算| 藏经阁中| 於我| 玩物| 充耳不闻| 力竭| 的几

Turn down the logging level a little bit

In [39]:
import sys
logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [40]:
token_ids = encode("归妹趋无妄", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['归妹趋无妄']

In [41]:
token_ids = encode("天之道 损有余而补不足", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['天之道', ' ', '损有余而补不足']

In [42]:
token_ids = encode("刷刷刷", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['刷刷刷']

In [46]:
token_ids = encode("阿弥陀佛", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['阿弥陀佛']

In [43]:

token_ids = encode("hello", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['h', 'e', 'l', 'l', 'o']

In [44]:
print(decode(encode(text[30000:31000], pair_to_code), vocab))

我搜看。”坚怒曰：“汝有何力，敢小觑我！”方欲交兵，刘表便退。坚纵马赶去，两山后伏兵齐起，背后蔡瑁、蒯越赶来，将孙坚困在垓心。正是：玉玺得来无用处，反因此宝动刀兵。毕竟孙坚怎地脱身，且听下文分解。



第七回 袁绍磐河战公孙 孙坚跨江击刘表

    		

    却说孙坚被刘表围住，亏得程普、黄盖、韩当三将死救得脱，折兵大半，夺路引兵回江东。自此孙坚与刘表结怨。

    且说袁绍屯兵河内，缺少粮草。冀州牧韩馥，遣人送粮以资军用。谋士逢纪说绍曰：“大丈夫纵横天下，何待人送粮为食！冀州乃钱粮广盛之地，将军何不取之？”绍曰：“未有良策。”纪曰：“可暗使人驰书与公孙瓒，令进兵取冀州，约以夹攻，瓒必兴兵。韩馥无谋之辈，必请将军领州事；就中取事，唾手可得。”绍大喜，即发书到瓒处。瓒得书，见说共攻冀州，平分其地，大喜，即日兴兵。

    绍却使人密报韩馥。馥慌聚荀谌、辛评二谋士商议。谌曰：“公孙瓒将燕、代之众，长驱而来，其锋不可当。兼有刘备、关、张助之，难以抵敌。今袁本初智勇过人，手下名将极广，将军可请彼同治州事，彼必厚待将军，无患公孙瓒矣。”韩馥即差别驾关纯去请袁绍。长史耿武谏曰：“袁绍孤客穷军，仰我鼻息，譬如婴儿在股掌之上，绝其乳哺，立可饿死。奈何欲以州事委之？此引虎入羊群也。”馥曰：“吾乃袁氏之故吏，才能又不如本初。古者择贤者而让之，诸君何嫉妒耶？”耿武叹曰：“冀州休矣！”于是弃职而去者三十余人。独耿武与关纯伏于城外，以待袁绍。

    数日后，绍引兵至。耿武、关纯拔刀而出，欲刺杀绍。绍将颜良立斩耿武，文丑砍死关纯。绍入冀州，以馥为奋威将军，以田丰、沮授、许攸、逢纪分掌州事，尽夺韩馥之权。馥懊悔无及，遂弃下家小，匹马往投陈留太守张邈去了。

    却说公孙瓒知袁绍已据冀州，遣弟公孙越来见绍，欲分其地。绍曰：“可请汝兄自来，吾有商议。”越辞归。行不到五十里，道旁闪出一彪军马，口称：“我乃董丞相家将也！”乱箭射死公孙越。从人逃回见公孙瓒，报越已死。瓒大怒曰：“袁绍诱我起兵攻韩馥，他却就里取事；今又诈董卓兵射死吾弟，此冤如何不报！”尽起本部兵，杀奔冀州来。

    绍知瓒兵至，亦领军出。二军会于磐河之上：绍军于磐河桥东，瓒军于桥西。瓒立马桥上，大呼曰：“背义之徒，何敢卖我！”绍亦策马至桥边，指瓒曰：“韩馥无才，愿让冀州于吾，与尔


In [45]:

token_ids = encode(text[30000:31000], pair_to_code=pair_to_code)
print("|".join(by_token_decode(token_ids, vocab=vocab)))

我|搜|看|。|”|坚|怒曰|：|“|汝有何|力|，|敢|小觑|我|！|”|方欲|交兵|，|刘表|便退|。|坚|纵马|赶去|，|两|山后|伏兵|齐|起|，|背后|蔡瑁|、|蒯越|赶来|，|将|孙坚|困在垓心|。|正是|：|玉玺|得|来|无用|处|，|反|因此|宝|动|刀|兵|。|毕竟|孙坚|怎|地|脱身|，|且听下文分解|。|



|第七|回| |袁绍|磐|河|战|公孙| |孙坚|跨|江|击|刘表|

    		|

    |却说|孙坚|被|刘表|围住|，|亏|得|程普|、|黄盖|、|韩当|三将|死|救|得脱|，|折兵大半|，|夺路|引兵|回江东|。|自此|孙坚|与|刘表|结|怨|。|

    |且说|袁绍|屯兵|河内|，|缺|少|粮草|。|冀州|牧|韩馥|，|遣人送|粮|以资|军|用|。|谋士|逢纪|说|绍曰|：|“|大丈夫|纵横天下|，|何|待人|送|粮|为食|！|冀州|乃|钱粮|广|盛|之地|，|将军何不|取之|？|”|绍曰|：|“|未有|良策|。|”|纪|曰|：|“|可|暗使人|驰|书与|公孙瓒|，|令|进兵|取|冀州|，|约|以|夹攻|，|瓒|必|兴兵|。|韩馥|无谋|之辈|，|必|请将军|领|州事|；|就|中取事|，|唾手可得|。|”|绍|大喜|，|即|发书|到|瓒|处|。|瓒|得书|，|见|说|共攻|冀州|，|平|分|其地|，|大喜|，|即日|兴兵|。|

    |绍|却|使人|密报|韩馥|。|馥|慌|聚|荀|谌|、|辛评|二|谋士|商议|。|谌曰|：|“|公孙瓒|将|燕|、|代|之众|，|长驱|而来|，|其锋|不可当|。|兼|有|刘备|、|关|、|张|助之|，|难以|抵敌|。|今|袁本初|智勇|过人|，|手下|名将|极广|，|将军可|请|彼|同|治|州事|，|彼必|厚待|将军|，|无患|公孙瓒|矣|。|”|韩馥|即差|别驾|关|纯|去|请|袁绍|。|长史|耿武|谏曰|：|“|袁绍|孤|客|穷|军|，|仰|我|鼻|息|，|譬|如|婴|儿|在|股|掌|之上|，|绝其|�|�|�|�|，|立|可|饿|死|。|奈何|欲以|州事|委|之|？|此|引|虎|入|羊|群|也|。|”|馥|曰|：|“|吾乃|袁氏|之故|吏|，|才|能|又|不如|本初|。|古|者|择|贤|者|而|让|之|，|诸君|何|嫉妒|耶|？|”|耿武|叹曰|：|“|冀州|休矣|！|”|于

In [46]:
decode(encode(text[:1000], pair_to_code), vocab) == text[:1000]

True